# Notebook 17 — Gradient-Based Optimization (A1 MLP + B6)

**Goal**: Directly optimize design parameters through back-propagation on the frozen
A1 surrogate. No inverse model is trained — parameters are optimised per target.

| Aspect | Detail |
|--------|--------|
| **Forward surrogate** | A1 MLP (frozen, from Notebook 11) |
| **Inverse method** | Direct gradient descent on params |
| **Optimiser** | Adam, 500 steps per target, 10 random restarts |
| **Loss** | Frequency-weighted spectral MSE |
| **Key advantage** | No training needed; exact optimisation; handles any target |

In [ ]:
# ============================================================
# Cell 1 — Imports & Configuration
# ============================================================
import sys, os, time, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

sys.path.insert(0, os.path.abspath('../src'))
from Theoretical_model import calculate_acoustic_properties
from physics_guided_CD_FiLM import PARAM_RANGES, validate_and_clip_parameters

assert torch.cuda.is_available(), 'CUDA required'
device = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print(f'Device: {torch.cuda.get_device_name()}')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths
DATA_PATH      = '../data/lhs_data_full_spectrum.npz'
MODEL_DIR      = '../models'
SURROGATE_PATH = os.path.join(MODEL_DIR, 'forward_surrogate_mlp.pth')
SCALER_PATH    = os.path.join(MODEL_DIR, 'forward_surrogate_mlp_scaler.pkl')

# Optimisation hyper-parameters
OPT_STEPS      = 500      # gradient steps per restart
OPT_LR         = 0.02     # Adam LR for parameter optim
NUM_RESTARTS   = 10       # random restarts
NUM_TEST       = 50       # how many test targets to optimise
NUM_PARAMS     = 20
NUM_FREQ       = 1000

PARAM_NAMES = ['d1','d2','d3','d4','d5','d6','d7','d8','d9','d10',
               'm2','m3','m5','m6','m8','m9','rho','eta','E','nu']

print('\n✓ Imports & config ready')

In [ ]:
# ============================================================
# Cell 2 — Load Data (for test targets & scaler)
# ============================================================
print('Loading NPZ …')
t0 = time.time()
raw = np.load(DATA_PATH)
params_all  = raw['params'].astype(np.float32)
spectra_all = raw['spectra'].astype(np.float32)
frequencies = raw['frequencies']
print(f'  Loaded in {time.time()-t0:.1f}s')

X_train, X_temp, Y_train, Y_temp = train_test_split(
    params_all, spectra_all, test_size=0.2, random_state=SEED)
X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.5, random_state=SEED)

scaler_x = MinMaxScaler()
scaler_x.fit(X_train)

print(f'  Test set: {X_test.shape[0]:,} samples')
print('✓ Data ready (only test targets needed)')

In [ ]:
# ============================================================
# Cell 3 — Load Forward Surrogate (frozen)
# ============================================================
class ForwardSurrogateMLP(nn.Module):
    def __init__(self, in_dim=20, out_dim=1000, hidden_dim=512, num_layers=4, dropout=0.05):
        super().__init__()
        layers = []
        prev = in_dim
        for _ in range(num_layers):
            layers += [nn.Linear(prev, hidden_dim), nn.BatchNorm1d(hidden_dim),
                       nn.LeakyReLU(0.01, inplace=True), nn.Dropout(dropout)]
            prev = hidden_dim
        layers += [nn.Linear(hidden_dim, out_dim), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

ckpt_surr = torch.load(SURROGATE_PATH, map_location=device, weights_only=False)
arch = ckpt_surr['architecture']
surrogate = ForwardSurrogateMLP(**arch).to(device)
surrogate.load_state_dict(ckpt_surr['model_state_dict'])
surrogate.eval()
for p in surrogate.parameters(): p.requires_grad = False
print(f'✓ Surrogate loaded (MSE={ckpt_surr["test_metrics"]["spectral_mse"]:.2e})')

In [ ]:
# ============================================================
# Cell 4 — Frequency Weights & Optim Function
# ============================================================
def build_freq_weights(num_freq=1000, low_cutoff=400, low_weight=2.0, device='cpu'):
    w = torch.ones(num_freq, device=device)
    w[:low_cutoff] = low_weight
    return w / w.mean()

freq_w = build_freq_weights(device=device)


def optimise_params(target_spec_tensor, surrogate_fn, num_restarts=10,
                    num_steps=500, lr=0.02, verbose=False):
    """
    Optimise normalised params (in [0,1]^20) to match a target spectrum.
    Uses multiple random restarts, returns best parameters.
    """
    target = target_spec_tensor.detach()  # (1, 1000) or (1000,)
    if target.dim() == 1:
        target = target.unsqueeze(0)

    best_loss = float('inf')
    best_params = None

    for restart in range(num_restarts):
        # Initialize in (0,1) using logit space for unconstrained optimisation
        raw = torch.randn(1, NUM_PARAMS, device=device) * 0.3  # logit-ish init
        raw = nn.Parameter(raw)
        opt = optim.Adam([raw], lr=lr)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=num_steps, eta_min=lr*0.01)

        for step in range(num_steps):
            params_01 = torch.sigmoid(raw)  # constrain to [0,1]
            pred_spec = surrogate_fn(params_01)
            loss = ((pred_spec - target)**2 * freq_w).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()

        # Final evaluation
        with torch.no_grad():
            params_01 = torch.sigmoid(raw)
            pred_spec = surrogate_fn(params_01)
            final_loss = ((pred_spec - target)**2).mean().item()

        if final_loss < best_loss:
            best_loss = final_loss
            best_params = params_01.detach().clone()

        if verbose and (restart + 1) % 5 == 0:
            print(f'    restart {restart+1}/{num_restarts}  best MSE={best_loss:.6e}')

    return best_params, best_loss

print('✓ Optimisation function defined')

In [ ]:
# ============================================================
# Cell 5 — Run Optimisation on Test Targets
# ============================================================
np.random.seed(77)
test_indices = np.random.choice(len(X_test), NUM_TEST, replace=False)

results = []
print(f'Optimising {NUM_TEST} test targets ({NUM_RESTARTS} restarts × {OPT_STEPS} steps each)…\n')
t_start = time.time()

for i, idx in enumerate(test_indices):
    target = torch.tensor(Y_test[idx], dtype=torch.float32, device=device)
    best_params, best_mse = optimise_params(
        target, surrogate, num_restarts=NUM_RESTARTS,
        num_steps=OPT_STEPS, lr=OPT_LR, verbose=False)

    results.append({
        'idx': idx, 'mse': best_mse,
        'params_norm': best_params.cpu().numpy().flatten(),
    })

    if (i + 1) % 10 == 0 or i == 0:
        elapsed = time.time() - t_start
        print(f'  [{i+1:3d}/{NUM_TEST}] MSE={best_mse:.6e}  '
              f'({elapsed:.0f}s elapsed, {elapsed/(i+1):.1f}s/target)')

total_time = time.time() - t_start
mse_arr = np.array([r['mse'] for r in results])
print(f'\n✓ Done in {total_time:.0f}s ({total_time/NUM_TEST:.1f}s/target)')
print(f'  Mean MSE  : {mse_arr.mean():.6e}')
print(f'  Median    : {np.median(mse_arr):.6e}')
print(f'  95th pct  : {np.percentile(mse_arr, 95):.6e}')

In [ ]:
# ============================================================
# Cell 6 — MSE Histogram
# ============================================================
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.log10(mse_arr + 1e-12), bins=30, edgecolor='k', alpha=0.7)
ax.axvline(np.log10(mse_arr.mean()), color='r', ls='--', label=f'Mean={mse_arr.mean():.2e}')
ax.set_xlabel('log₁₀(MSE)'); ax.set_ylabel('Count')
ax.set_title(f'Gradient Optim (A1+B6) — MSE Distribution ({NUM_TEST} targets)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 7 — Overlay: 6 Spectra
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
for ax, r in zip(axes.flat, results[:6]):
    target = Y_test[r['idx']]
    params_t = torch.tensor(r['params_norm'], dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        recon = surrogate(params_t).cpu().numpy().flatten()

    ax.plot(frequencies, target, 'b-',  lw=1.2, label='Target')
    ax.plot(frequencies, recon,  'r--', lw=1.0, label='Grad Optim')
    ax.set_title(f'MSE={r["mse"]:.2e}', fontsize=10)
    ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.supxlabel('Frequency (Hz)')
fig.supylabel('Absorption Coefficient')
fig.suptitle('Gradient Optim (A1+B6) — Reconstructions', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 8 — Convergence Trace (1 target)
# ============================================================
# Re-run one optimisation storing loss history
idx0 = test_indices[0]
target_t = torch.tensor(Y_test[idx0], dtype=torch.float32, device=device).unsqueeze(0)

raw = torch.randn(1, NUM_PARAMS, device=device) * 0.3
raw = nn.Parameter(raw)
opt = optim.Adam([raw], lr=OPT_LR)
sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=OPT_STEPS, eta_min=OPT_LR*0.01)

history = []
for step in range(OPT_STEPS):
    params_01 = torch.sigmoid(raw)
    pred = surrogate(params_01)
    loss = ((pred - target_t)**2 * freq_w).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()
    sched.step()
    history.append(loss.item())

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(history)
ax.set_xlabel('Step'); ax.set_ylabel('Weighted MSE (log)')
ax.set_title('Convergence Trace (single restart)')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# Cell 9 — TMM Validation (5 samples)
# ============================================================
NUM_TMM = 5
fig, axes = plt.subplots(1, NUM_TMM, figsize=(20, 4), sharex=True, sharey=True)
tmm_results = []

for ax, r in zip(axes, results[:NUM_TMM]):
    target = Y_test[r['idx']]
    pred_raw = scaler_x.inverse_transform(r['params_norm'].reshape(1, -1)).flatten()
    pred_raw = validate_and_clip_parameters(pred_raw.reshape(1, -1)).flatten()

    param_dict = {
        'rho': float(pred_raw[16]), 'eta': float(pred_raw[17]),
        'E': float(pred_raw[18]), 'nu': float(pred_raw[19]), 'W': 2000.0,
        'd': [float(pred_raw[i]) for i in range(10)],
        'm': {2: float(pred_raw[10]), 3: float(pred_raw[11]),
              5: float(pred_raw[12]), 6: float(pred_raw[13]),
              8: float(pred_raw[14]), 9: float(pred_raw[15])}
    }
    _, alpha_tmm, _, _ = calculate_acoustic_properties(param_dict)

    mse_tmm = np.mean((alpha_tmm - target)**2)
    avg_err = abs(alpha_tmm.mean() - target.mean())
    tmm_results.append({'idx': r['idx'], 'mse': mse_tmm, 'avg_err': avg_err})

    ax.plot(frequencies, target,    'b-',  lw=1.2, label='Target')
    ax.plot(frequencies, alpha_tmm, 'r--', lw=1.0, label='TMM(pred θ)')
    ax.set_title(f'MSE={mse_tmm:.2e}\nΔavg={avg_err:.4f}', fontsize=9)
    ax.set_ylim(-0.02, 1.02); ax.grid(True, alpha=0.3)
    if r == results[0]: ax.legend(fontsize=8)

fig.supxlabel('Frequency (Hz)')
fig.suptitle('Gradient Optim (A1+B6) — TMM Validation', fontsize=13)
plt.tight_layout(); plt.show()

print('\n  TMM Validation:')
for r in tmm_results:
    print(f'    Sample {r["idx"]:6d}  MSE={r["mse"]:.4e}  Δavg={r["avg_err"]:.4f}')
print(f'  Mean TMM MSE: {np.mean([r["mse"] for r in tmm_results]):.4e}')

---
## Summary

| Metric | Value |
|--------|-------|
| Method | Direct gradient optimisation through frozen surrogate |
| Per-target cost | ~500 steps × 10 restarts (~seconds on GPU) |
| No training | Zero training time; works for any target |

**Key advantage:** No model to train; can optimise for arbitrary custom targets.  
**Limitation:** Slow per-query inference (seconds vs. milliseconds for learned models).